In [0]:
# Change the course name as per your course name
COURSE_NAME = "Get Started with Databricks for Data Warehousing"
RUN_QA_CHECKER = True
QA_TASK_RELATIVE_PATH = "./qa_content_checker"
QA_TASK_NAME = "QA Content Checker — Grammar, Deprecated, UI Steps"

# Tester emails — notified on job failure and success. Add as many as you like with comma seperated emails
TESTER_EMAILS = [
    "gourav.prajapati@databricks.com"
]

# Lab notebook paths for the auto-fill step (fills <FILL_IN> placeholders before a run).
# Left EMPTY for this course on purpose: this runner executes the classroom SETUP files
# (see COURSE_TASKS), not the demo/lab notebooks, because the demos/labs are UI-driven and
# cannot be validated by a headless job. There is nothing to auto-fill for a run we don't
# perform. If the labs are ever run as tasks, list their paths here.
LAB_NOTEBOOKS = []

# ── Compute Type Registry ─────────────────────────────────────────────────────
# Define all compute configurations available to tasks in this course.
# Each key becomes the compute_type value you assign to a task in COURSE_TASKS.
# Cell 14 reads these entries to create the correct compute resource per task.
#
# Supported "type" values:
#   "serverless"      – Databricks Serverless. Requires: environment_version (str)
#   "classic_cluster" – Existing all-purpose cluster. Requires: cluster_id (str)
#   "new_cluster"     – New cluster per run. Requires: cluster_config (dict)
#                       Keys map directly to ClusterSpec fields:
#                       spark_version, node_type_id, num_workers, etc.
#
# Tasks that omit compute_type fall back to DEFAULT_COMPUTE_TYPE.
# ─────────────────────────────────────────────────────────────────────────────

DEFAULT_COMPUTE_TYPE = "serverless_v5"

COMPUTE_TYPES = {
    "serverless_v5": {
        "type": "serverless",
        "environment_version": "5",
    },
    # ── SQL Warehouse ─────────────────────────────────────────────────────────
    # Use for notebooks that require a SQL warehouse (e.g. SQL Scripting,
    # Stored Procedures). The job will find an existing warehouse by name or
    # create one automatically using the config below.
    # Only PRO / Serverless warehouses are supported — not Classic.
    "shared_warehouse": {
        "type": "warehouse",
        "warehouse_name": "shared_warehouse",
        "cluster_size": "2X-Small",
        "warehouse_type": "PRO",
        "auto_stop_mins": 30,
    },
    # "serverless_v4": {
    #     "type": "serverless",
    #     "environment_version": "4",
    # },
    # "my_classic_cluster": {
    #     "type": "classic_cluster",
    #     "cluster_id": "<your-cluster-id>",
    # },
    # "new_standard_cluster": {
    #     "type": "new_cluster",
    #     "cluster_config": {
    #         "spark_version": "15.4.x-scala2.12",
    #         "node_type_id":  "i3.xlarge",
    #         "num_workers":   2,
    #     },
    # },
}

# Define the course notebook DAG here so this can be used for job scheduling as per your course.
#
# WHAT THIS RUNNER TESTS FOR THIS COURSE (important — HYBRID model):
#   This course splits into two kinds of content, verified by inspecting each notebook:
#
#   1. DEMOS (01, 02, 04, 06, 08, 10) — run the DEMO NOTEBOOK itself.
#      These contain real, pre-written SQL (SELECT / SHOW / DESCRIBE HISTORY / OPTIMIZE /
#      star-schema builds / MEASURE queries) that executes end to end. Each demo self-
#      provisions by running its own `%run ./Includes/Classroom-Setup-NN` at the top, so
#      running the demo also exercises its setup. Testing these validates real functionality.
#      (Demos 02 and 08 contain one "run what Genie generated" cell, but it is a self-contained
#      no-op with no downstream cell depending on it, so the notebook still runs clean.)
#
#   2. LABS (03, 05, 07, 09, 11) — run the CLASSROOM SETUP file only.
#      This is a "Genie Code" course: the lab answer cells are intentionally EMPTY except for a
#      `-- Use Genie Code (Cmd/Ctrl + I) ...` comment; the learner generates the SQL
#      interactively. There is no <FILL_IN> and no stored solution to auto-fill, so those cells
#      run as no-ops AND later cells depend on objects the learner was meant to create (e.g.
#      lab 11's `SELECT ... FROM vw_store_metrics` needs a view an empty Genie cell should have
#      created). Running the lab notebook headlessly would therefore FAIL on those downstream
#      cells. So we test the lab's Classroom-Setup instead — the testable, code-only part.
#
#   The setup files are pure code: each runs `%run ./Classroom-Setup-Common` (provisions the
#   per-user catalog + `vintage_audio` schema, creates a data volume, stages the course CSVs)
#   and `%run ./state_helpers` (ensure_* functions), then calls the ensure_* it needs
#   (ensure_source_dimensions / ensure_orders_bronze_full / ensure_star_schema /
#   ensure_metric_view_product). Auto Loader ingestion uses trigger(availableNow) — finite,
#   does not hang.
#
# Dependency rationale:
#   • Tasks are chained sequentially in course order with run_if=ALL_DONE. Ordering is not
#     strictly required for correctness (each demo's setup and each ensure_* self-provisions),
#     but everything writes into the SAME per-user `vintage_audio` schema and data volume, so
#     serializing avoids write races on shared tables/checkpoints and gives a clean run where
#     one failure does not skip the rest.
#   • All tasks run on Serverless (manifest: supported_dbrs = Serverless).
#   • The UI-only "00 - Course Overview", "AGENDA", and "Summary and Next Steps" are excluded.
COURSE_TASKS = [
    {
        "task_key": "demo_01",
        "name": "01 Demo - Databricks Workspace Walkthrough",
        "relative_path": "../../01 Demo - Databricks Workspace Walkthrough",
        "depends_on": [],
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "demo_02",
        "name": "02 Demo - Querying the Vintage Audio Warehouse",
        "relative_path": "../../02 Demo - Querying the Vintage Audio Warehouse",
        "depends_on": ["demo_01"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "lab_03_setup",
        "name": "Classroom-Setup-03 (for 03 Lab - Explore the Vintage Audio Tables) — lab is Genie-driven, testing setup only",
        "relative_path": "../Classroom-Setup-03",
        "depends_on": ["demo_02"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "demo_04",
        "name": "04 Demo - Tuning Unity Catalog managed Tables for Analytical Queries",
        "relative_path": "../../04 Demo - Tuning Unity Catalog managed Tables for Analytical Queries",
        "depends_on": ["lab_03_setup"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "lab_05_setup",
        "name": "Classroom-Setup-05 (for 05 Lab - Optimize the Products Table) — lab is Genie-driven, testing setup only",
        "relative_path": "../Classroom-Setup-05",
        "depends_on": ["demo_04"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "demo_06",
        "name": "06 Demo - Loading Data into the Warehouse",
        "relative_path": "../../06 Demo - Loading Data into the Warehouse",
        "depends_on": ["lab_05_setup"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "lab_07_setup",
        "name": "Classroom-Setup-07 (for 07 Lab - Incrementally Load New Orders) — lab is Genie-driven, testing setup only",
        "relative_path": "../Classroom-Setup-07",
        "depends_on": ["demo_06"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "demo_08",
        "name": "08 Demo - Building a Star Schema",
        "relative_path": "../../08 Demo - Building a Star Schema",
        "depends_on": ["lab_07_setup"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "lab_09_setup",
        "name": "Classroom-Setup-09 (for 09 Lab - Build a Store Dimension) — lab is Genie-driven, testing setup only",
        "relative_path": "../Classroom-Setup-09",
        "depends_on": ["demo_08"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "demo_10",
        "name": "10 Demo - Deliver Insights with Metric Views",
        "relative_path": "../../10 Demo - Deliver Insights with Metric Views",
        "depends_on": ["lab_09_setup"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "lab_11_setup",
        "name": "Classroom-Setup-11 (for 11 Lab - Create a Metric View for Store Performance) — lab is Genie-driven, testing setup only",
        "relative_path": "../Classroom-Setup-11",
        "depends_on": ["demo_10"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
    {
        "task_key": "capstone_setup",
        "name": "Classroom-Setup-Capstone (for Capstone Lab - Vintage Audio Quarterly Performance Review) — lab is Genie-driven, testing setup only",
        "relative_path": "../Classroom-Setup-Capstone",
        "depends_on": ["lab_11_setup"],
        "run_if": "ALL_DONE",
        "compute_type": "serverless_v5",
    },
]

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Utility: wrap_failing_sql_cell / restore_sql_cell
#
# Purpose
# -------
# Some demo notebooks contain SQL cells that are *intentionally designed to fail*
# (e.g. to demonstrate BEGIN...END halting behaviour). When those notebooks run
# as job tasks the cell failure stops the entire notebook, skipping all
# subsequent cells (cleanup, further demos, etc.).
#
# These helpers let the course tester temporarily wrap any such cell with an
# outer BEGIN...DECLARE EXIT HANDLER...END block — pure SQL, no language change
# — so the notebook continues past the intentional failure. After the job
# completes the original source is restored so students always see the clean
# demo notebook.
#
# How to use
# ----------
# 1. Before job creation / run:
#       restores = patch_notebooks_for_job(CELLS_TO_PATCH)
#
# 2. After job completes (pass/fail):
#       restore_notebooks_after_job(restores)
#
# CELLS_TO_PATCH is a list of dicts defined in the cell that follows this one.
# Each entry:
#   {
#       "notebook_path" : absolute workspace path to the notebook,
#       "cell_nuid"     : nuid of the SQL cell to wrap,
#       "handler_message": (optional) text returned by the handler SELECT,
#   }
# ─────────────────────────────────────────────────────────────────────────────

import base64, json, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

_sdk_client = None

def _get_client(w=None):
    """Returns a WorkspaceClient, reusing a module-level instance if available."""
    if w:
        return w
    global _sdk_client
    if _sdk_client is None:
        _sdk_client = WorkspaceClient()
    return _sdk_client


def wrap_failing_sql_cell(
    notebook_path: str,
    cell_nuid: str,
    handler_message: str = None,
    w=None,
) -> str:
    """
    Wraps a SQL cell's BEGIN...END block with an outer DECLARE EXIT HANDLER
    so the notebook continues executing even if the inner block fails.
    The cell stays SQL — no language change.

    Args:
        notebook_path   : Absolute workspace path to the target notebook.
        cell_nuid       : nuid of the SQL cell to wrap.
        handler_message : Optional message shown as a result row when the
                          exception is caught.
        w               : WorkspaceClient to reuse (optional).
    Returns:
        original_source (str) — pass this to restore_sql_cell() to undo.
    """
    _w  = _get_client(w)
    msg = handler_message or (
        "[Expected failure] The inner BEGIN...END block failed as designed. "
        "Notebook continues."
    )

    # --- export
    export  = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb      = json.loads(base64.b64decode(export.content))

    original_source = None
    for cell in nb.get("cells", []):
        if cell.get("metadata", {}).get("nuid", "") == cell_nuid:
            original_source = (
                "".join(cell["source"])
                if isinstance(cell["source"], list)
                else cell["source"]
            )
            indented = "\n".join("  " + line for line in original_source.splitlines())
            cell["source"] = (
                "-- Outer block catches the expected error; inner block is unchanged.\n"
                "BEGIN\n"
                "  DECLARE EXIT HANDLER FOR SQLEXCEPTION\n"
                f"    SELECT '{msg}' AS message;\n\n"
                "  BEGIN\n"
                f"{indented}\n"
                "  END;\n"
                "END;"
            )
            break

    if original_source is None:
        raise ValueError(f"Cell nuid='{cell_nuid}' not found in: {notebook_path}")

    # --- re-import
    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Wrapped  : {notebook_path.split('/')[-1]}  (nuid: {cell_nuid[:8]}...)")
    return original_source


def restore_sql_cell(
    notebook_path: str,
    cell_nuid: str,
    original_source: str,
    w=None,
):
    """
    Restores a cell to its original source. Reverses wrap_failing_sql_cell().

    Args:
        notebook_path  : Absolute workspace path to the target notebook.
        cell_nuid      : nuid of the cell to restore.
        original_source: The string returned by wrap_failing_sql_cell().
        w              : WorkspaceClient to reuse (optional).
    """
    _w     = _get_client(w)
    export = _w.workspace.export(path=notebook_path, format=ExportFormat.JUPYTER)
    nb     = json.loads(base64.b64decode(export.content))

    for cell in nb.get("cells", []):
        if cell.get("metadata", {}).get("nuid", "") == cell_nuid:
            cell["source"] = original_source
            break

    encoded = base64.b64encode(json.dumps(nb).encode()).decode()
    _w.workspace.import_(
        path=notebook_path,
        format=ImportFormat.JUPYTER,
        overwrite=True,
        content=encoded,
    )
    print(f"  ✅ Restored : {notebook_path.split('/')[-1]}  (nuid: {cell_nuid[:8]}...)")


def patch_notebooks_for_job(patches: list, w=None) -> list:
    """
    Wraps all cells listed in `patches` before a job run.
    Returns a restore-list to pass to restore_notebooks_after_job().

    Args:
        patches : list of dicts, each with keys:
                    notebook_path (str)       — absolute workspace path
                    cell_nuid     (str)       — nuid of the failing cell
                    handler_message (str, opt)— custom handler message
        w       : WorkspaceClient to reuse (optional).
    Returns:
        restores (list) — same dicts with 'original_source' added.
    """
    print("\n═" * 36 + "\nPATCHING NOTEBOOKS FOR JOB RUN\n" + "═" * 36)
    restores = []
    for p in patches:
        original = wrap_failing_sql_cell(
            notebook_path   = p["notebook_path"],
            cell_nuid       = p["cell_nuid"],
            handler_message = p.get("handler_message"),
            w               = w,
        )
        restores.append({**p, "original_source": original})
    print(f"  {len(restores)} cell(s) patched.\n")
    return restores


def restore_notebooks_after_job(restores: list, w=None):
    """
    Restores all cells patched by patch_notebooks_for_job().

    Args:
        restores : the list returned by patch_notebooks_for_job().
        w        : WorkspaceClient to reuse (optional).
    """
    print("\n═" * 36 + "\nRESTORING NOTEBOOKS AFTER JOB RUN\n" + "═" * 36)
    for r in restores:
        restore_sql_cell(r["notebook_path"], r["cell_nuid"], r["original_source"], w=w)
    print(f"  {len(restores)} cell(s) restored.\n")


print("✅ Utility functions loaded: wrap_failing_sql_cell, restore_sql_cell,"
      " patch_notebooks_for_job, restore_notebooks_after_job")